# Training on SSC: do the networks use the order of events, and what does it cost?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Avi-Seth/UCLA-COSMOS-SNN-Energy-efficiency/blob/main/SHD-SSC-order-of-events/03_train_SSC.ipynb)

**The question.** A spoken command in SSC (Spiking Speech Commands, 35 English words) is a list of
spikes on 700 channels. We cut every recording into time windows and count the spikes of every channel
in every window. This gives a table with one row per window. Some networks read the rows **in order**
(spiking networks, a recurrent network, an LSTM). Do they really use that order, or would a network
that never sees the order do just as well?

Three kinds of model read the same tables:

- the **count classifier** sees only each channel's total number of spikes over the whole recording;
- **order-blind designs** look at every window but cannot see the order of the windows. The best of
  them, chosen on held-out recordings, is the **order-blind ceiling r**;
- **order-using networks** read the windows in order. Each one is trained twice from the same seed:
  with the windows in order (test accuracy **a0**) and with every recording's windows shuffled (**a1**).

```
drop = a0 - a1 = (a0 - r) + (r - a1) = gain + deficit
```

This notebook trains the models on SSC and saves their results, including an energy estimate for every
model (computed, not measured). It does not depend on `02_train_SHD.ipynb`, so the two can run at the
same time in two Colab sessions if your Colab account allows two GPU sessions at once.
`04_analyze_results.ipynb` makes the tables and figures.

**What it runs.** Every recording is cut into 100 windows of 10 ms. 8 order-blind designs; the SSC
authors' LSTM set-up, lif_std, rlif and rnn; three seeds; 48 runs in all, about 15 to 16 hours on an
NVIDIA L4 GPU (a planning figure; the notebook times every model on your GPU and prints its own
estimate before training, and a progress bar then shows the time left).

**How to run it.** In Colab choose a GPU (Runtime > Change runtime type), then Runtime > Run all.
Nothing else is needed: the SSC files (a 1.5 GB download) are downloaded from the dataset authors'
site. As it is set, the notebook runs a **quick test** that uses a few recordings, so its numbers mean
nothing. For the full study, set `QUICK = False` in Step 1 and also choose High-RAM (Runtime > Change
runtime type): the 75,466 training recordings take about 5 GB of memory as 8-bit counts. Colab then
asks once for access to your Google Drive, and every finished run is saved there at once. A full run
can last longer than a Colab session; if the session stops, open the notebook again and run all the
cells: runs already saved are skipped.

## Step 1. Settings

- `QUICK = True` tests the whole notebook with a few recordings and one training epoch; its numbers
  mean nothing. `QUICK = False` runs the full study.
- `USE_DRIVE = True` keeps the results on your Google Drive. A full run always does, because a Colab
  machine and everything on it is deleted when the session ends.
- `MY_DATA_DIR` is optional: a folder that already holds `ssc_train.h5` and `ssc_test.h5` (or their
  `.h5.gz` versions), anywhere inside it. When it is empty, the files are downloaded.

In [ ]:
QUICK = True              # True: a quick test (its numbers mean nothing). False: the full study
USE_DRIVE = False         # True: keep the results on your Google Drive (a full run always does)
MY_DATA_DIR = ''          # optional: a folder with the SSC files, for example '/content/drive/MyDrive/data/'

RUN_CHECKS = True         # the self-checks before training (they take seconds)

## Step 2. Folders, Google Drive and snnTorch

The results of a full run go to Google Drive, so that they outlast the Colab session; Colab asks once
for permission. A quick test keeps everything on the Colab machine, and its last cell offers the
results as a download. snnTorch is a library of spiking neurons; it is used only by the self-checks,
to confirm that the spiking neurons written out in this notebook behave exactly like snnTorch's.

In [ ]:
if not QUICK and not USE_DRIVE:               # a full run needs a place that outlasts the session
    USE_DRIVE = True
    print('A full run keeps its results on Google Drive, so USE_DRIVE is set to True.')
if USE_DRIVE or MY_DATA_DIR.startswith('/content/drive'):   # Google Drive is needed: connect it
    from google.colab import drive            # the tool that connects Google Drive to this notebook
    drive.mount('/content/drive')             # your Drive files appear under /content/drive/MyDrive

if USE_DRIVE:
    BASE_DIR = '/content/drive/MyDrive/shd_ssc_order/'    # on your Drive: kept after the session
else:
    BASE_DIR = '/content/shd_ssc_order/'                  # on this Colab machine: deleted after the session
if QUICK:
    RESULTS_DIR = BASE_DIR + 'results_quick/'             # a quick test writes to its own folder
else:
    RESULTS_DIR = BASE_DIR + 'results/'                   # where every finished run is saved
DATA_DIR = '/content/data/'                               # the data files, on this machine's own disk (fast)
DATA_URL = 'https://zenkelab.org/datasets/'               # the dataset authors' site
print('results folder:', RESULTS_DIR)
%pip install -q snntorch==1.0.0

## Step 3. Libraries

The tools this notebook uses. The last lines check that a GPU is available.

In [ ]:
import os                                     # folders and file names
import gzip                                   # unpacking the downloaded .gz files
import json                                   # saving results as text files
import time                                   # measuring how long things take
import shutil                                 # copying files and making .zip files
import hashlib                                # checksums of the downloaded files
import urllib.request                         # downloading files

import h5py                                   # reading the .h5 data files
import numpy as np                            # arrays of numbers
import torch                                  # the neural-network library
import torch.nn as nn                         # ready-made network layers
from tqdm.auto import tqdm                    # progress bars

if torch.cuda.is_available():                 # is there a GPU?
    DEVICE = 'cuda'                           # yes: compute on it
    print('GPU:', torch.cuda.get_device_name(0))
else:
    DEVICE = 'cpu'                            # no: training would be very slow
    print('No GPU. In Colab choose Runtime > Change runtime type > a GPU, then run the notebook again.')

## Step 4. The fixed numbers of the study

Every model is trained with three different seeds (a seed fixes the random starting weights). Every
model uses the same recipe: Adamax at a learning rate of 0.002, 60 epochs, the last epoch kept (Adamax,
as the SSC authors' LSTM).

In [ ]:
C = 700                        # input channels (the 700 ear-model channels)
K = 35                         # classes: 35 spoken English words
H = 128                        # hidden units of the small models
BATCH = 128                    # recordings per training step
T = 100                        # windows per recording
WIDTH = 0.010                  # window width in seconds (10 ms)
SEEDS = [13, 29, 43]           # three seeds: every model is trained three times
HOLDOUT = 0.05                 # 5% of the training recordings are held out to choose designs
VAL_SEED = 2027                # seed of the random choice of held-out recordings
EVAL_SEED = 17                 # EVAL_SEED + seed is the seed used when testing
SHUFFLE_SEED_FIT = 1000        # + 10 x seed: window orders of the shuffled fitting recordings
SHUFFLE_SEED_VAL = 1500        # + 10 x seed: ... of the shuffled held-out recordings
SHUFFLE_SEED_TEST = 2000       # + 10 x seed: ... of the shuffled test recordings
TRAIN_CHECK = 5000             # training accuracy is measured on 5,000 fitting recordings spread evenly
BETA = 0.9 ** (10 / 28)        # spiking neurons keep about 96.3% of their voltage per 10 ms window: the same
                               # memory in milliseconds as 90% per window at SHD's 25 windows (about 28 ms)
RECIPE = {'optimizer': 'adamax', 'lr': 0.002, 'epochs': 60, 'keep': 'final'}   # every model

DESIGNS = ['count_256', 'perminv', 'pool_128', 'pool_512', 'meanmax_deep_128',
           'meanmax_deep_128_d20', 'meanmax_deep_128_d50', 'meanmax_deep_128_d70']   # the 8 order-blind designs
NETWORKS = ['lstm_paper', 'lif_std', 'rlif', 'rnn']                                    # the order-using networks

if QUICK:                                     # a quick test: two seeds, one epoch, few recordings
    SEEDS_USED = SEEDS[:2]
    RECIPE['epochs'] = 1
    TRAIN_CHECK = 128
else:
    SEEDS_USED = SEEDS

## Step 5. Get the SSC files

The study uses two SSC files, `ssc_train.h5` (75,466 recordings) and `ssc_test.h5` (20,382
recordings); downloaded, they are 1.5 GB together, and unpacked 2.7 GB and 0.7 GB.

For every file the notebook looks, in this order:

1. on this machine, from earlier in the same session;
2. in `MY_DATA_DIR`, if it is set (the file or its `.gz` version, anywhere inside that folder);
3. on the dataset authors' site, https://zenkelab.org/datasets/. The compressed file is downloaded,
   checked against the checksum the site publishes (an MD5 checksum is a fingerprint of a file, so a
   damaged download is caught), and unpacked.

The files go to the Colab machine's own disk, which is fast to read. Every file is written under a
temporary name and renamed only when complete, so an interrupted copy or download is never used.

In [ ]:
def find_in_my_data(name):
    """The path of `name` (or of `name` + '.gz') anywhere inside MY_DATA_DIR, or None if it is not there."""
    if MY_DATA_DIR == '':                                              # no folder of your own was given
        return None
    for folder, subfolders, files in os.walk(MY_DATA_DIR):            # every folder inside MY_DATA_DIR
        subfolders.sort()                                              # look through them in alphabetical order
        for candidate in [name, name + '.gz']:                         # the file itself, or its compressed version
            if candidate in files:
                return os.path.join(folder, candidate)
    return None


def download(url, path):
    """Download `url` to `path` with a progress bar; the file gets its real name only when complete."""
    with urllib.request.urlopen(url, timeout=60) as response:          # open the connection
        size = int(response.headers.get('Content-Length', 0))          # the file's size, if the site gives it
        with open(path + '.part', 'wb') as out:
            with tqdm(total=size, unit='B', unit_scale=True, desc='downloading ' + os.path.basename(path)) as bar:
                while True:
                    chunk = response.read(4 * 1024 * 1024)             # 4 MB at a time
                    if len(chunk) == 0:                                # nothing left: the download is complete
                        break
                    out.write(chunk)
                    bar.update(len(chunk))
    os.replace(path + '.part', path)                                   # the real name, now that it is complete


def md5_of(path):
    """The MD5 checksum of a file: a fingerprint that changes if a single byte of the file changes."""
    digest = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(16 * 1024 * 1024)                           # 16 MB at a time
            if len(chunk) == 0:
                break
            digest.update(chunk)
    return digest.hexdigest()


def published_md5(dataset):
    """The checksums the dataset authors publish for 'shd' or 'ssc', as {file name: checksum}."""
    with urllib.request.urlopen(DATA_URL + dataset + '/md5sums.txt', timeout=60) as response:
        text = response.read().decode()
    sums = {}
    for line in text.splitlines():                                     # every line is 'checksum  file name'
        parts = line.split()
        if len(parts) == 2:
            sums[parts[1]] = parts[0]
    return sums


def unpack(packed, path):
    """Unpack a .gz file to `path`; the file gets its real name only when complete."""
    with gzip.open(packed, 'rb') as source:
        with open(path + '.part', 'wb') as out:
            shutil.copyfileobj(source, out, 16 * 1024 * 1024)          # 16 MB at a time
    os.replace(path + '.part', path)


def get_data_file(name):
    """Put the data file `name` (for example 'shd_train.h5') on this machine, and return where it is."""
    path = DATA_DIR + name
    if os.path.exists(path):                                           # 1. here from earlier in this session
        return path
    mine = find_in_my_data(name)
    if mine is not None:                                               # 2. in MY_DATA_DIR
        print('taking', name, 'from', mine)
        if mine.endswith('.gz'):
            unpack(mine, path)                                         # a compressed file: unpack it here
        else:
            shutil.copy(mine, path + '.part')                          # copy it here, under a temporary name
            os.replace(path + '.part', path)
        return path
    dataset = name.split('_')[0]                                       # 3. from the dataset authors' site
    packed = path + '.gz'                                              # the site has compressed files only
    url = DATA_URL + dataset + '/' + name + '.gz'
    try:
        download(url, packed)
    except Exception as error:                                         # no connection, a moved file, ...
        raise RuntimeError(f'{name}.gz could not be downloaded from {url} ({error}). Download the '
                           f'{dataset.upper()} files from https://zenkelab.org/datasets/ yourself, put them in a '
                           f'folder on your Google Drive, set MY_DATA_DIR in Step 1 to that folder, and run '
                           f'the notebook again.') from None
    try:
        expected = published_md5(dataset).get(name + '.gz')            # the checksum the site publishes
    except Exception:
        expected = None                                                # the list of checksums could not be read
    if expected is None:
        print('note: no published checksum was found for', name + '.gz', '- the download could not be checked')
    elif md5_of(packed) != expected:                                   # a damaged download
        os.remove(packed)
        raise RuntimeError(f'{name}.gz was damaged while downloading (its checksum is wrong). '
                           f'Run this cell again.')
    else:
        print(name + '.gz: its checksum matches the one the dataset authors publish')
    print('unpacking', name + '.gz')
    unpack(packed, path)
    os.remove(packed)                                                  # the compressed file is no longer needed
    return path


os.makedirs(DATA_DIR, exist_ok=True)                                   # create the data folder if needed
SSC_TRAIN = get_data_file('ssc_train.h5')                              # the training recordings
SSC_TEST = get_data_file('ssc_test.h5')                                # the test recordings
print('SSC files on this machine:', SSC_TRAIN, SSC_TEST)

## Step 6. From spikes to a table of counts

Every recording is a list of spike times (in seconds) and channels (0 to 699). We cut it into 100
windows of 10 ms from time 0 and count, for every window and every channel, how many spikes fell
there. A spike after the last window (after 1 s) is counted in the last window.

The files store spike times in half precision (16-bit numbers); they are converted to double precision
before any arithmetic, because in half precision some spikes would land in a neighbouring window. The
counts are stored as 8-bit numbers (0 to 255) to save memory; a count above 255 would be cut to 255,
and the cell tells you if that ever happened.

In [ ]:
def count_spikes(path, every=1):
    """Count the spikes of the recordings of one SSC file in 100 windows of 10 ms per channel: every
    recording, or only every `every`-th recording (a quick test).
    Returns the table (recordings x 100 x 700, 8-bit) and the classes."""
    with h5py.File(path, 'r') as f:                               # open the file for reading
        times = f['spikes']['times']                              # every recording's spike times (seconds)
        units = f['spikes']['units']                              # every spike's channel (0 to 699)
        labels = np.array(f['labels']).astype(np.int64)           # every recording's class (0 to 34)
        rows = np.arange(0, len(labels), every)                   # the recordings counted (all when every = 1)
        labels = labels[rows]
        n = len(rows)                                             # how many recordings are counted
        X = np.zeros((n, T, C), dtype=np.uint8)                   # the table, filled below
        clipped = 0                                               # cells whose count was above 255
        for start in tqdm(range(0, n, 1024), desc='counting spikes in ' + os.path.basename(path)):
            first, last = rows[start], rows[min(start + 1024, n) - 1]   # the first and last recording of the block
            block_times = times[first:last + 1:every]             # 1,024 recordings at a time: reading many at
            block_units = units[first:last + 1:every]             # once is much faster than one by one
            for k in range(len(block_times)):                     # one recording at a time
                t = block_times[k].astype(np.float64)             # its spike times, in double precision
                ch = block_units[k].astype(np.int64)              # its spike channels
                window = (t / WIDTH).astype(np.int64)             # window number = time / width
                window = window.clip(0, T - 1)                    # spikes after the last window are counted in it
                cell = window * C + ch                            # one number per (window, channel) cell
                counts = np.bincount(cell, minlength=T * C).reshape(T, C)   # spikes in every cell
                clipped += int((counts > 255).sum())
                X[start + k] = np.minimum(counts, 255)            # stored as 8-bit numbers
    print(f'{os.path.basename(path)}: {n} recordings, cells above 255 (cut to 255): {clipped}')
    return X, labels

## Step 7. Helper functions for the data

- `holdout` sets aside 5% of every class of the training recordings. The held-out recordings choose
  the best design; the test recordings never choose anything.
- `scaling_stats` computes, for every channel, the mean and the spread (standard deviation) of its
  counts on the fitting recordings, adding up 2,000 recordings at a time so that no large copy of the
  table is made. Most models subtract the mean and divide by the spread.
- `input_statistics` measures how many spikes and non-empty cells a test recording has on average; the
  energy estimate needs them.
- `window_orders` makes a random order of the windows for every recording (each recording its own
  order, by sorting random numbers), so the order carries no information a network could learn.
- `to_gpu` copies recordings to the GPU as 8-bit counts; `get_batch` takes a batch of them, each in its
  own window order when one is given, and turns the counts into float numbers.

In [ ]:
def holdout(y, share, seed):
    """Hold out round(share x class size) recordings of every class (at least one), chosen at random.
    Returns the rows kept for fitting and the held-out rows."""
    rng = np.random.default_rng(seed)                     # a random generator with a fixed seed
    held = []                                             # the held-out rows, class by class
    for k in range(K):                                    # every class
        rows_k = np.where(y == k)[0]                      # the recordings of class k
        n_k = max(1, int(round(share * len(rows_k))))     # how many of them to hold out
        held.append(rng.choice(rows_k, n_k, replace=False))   # choose them at random
    held = np.sort(np.concatenate(held))                  # all held-out rows, in increasing order
    fit = np.setdiff1d(np.arange(len(y)), held)           # every other row is used for fitting
    return fit, held


def scaling_stats(X, rows):
    """Per-channel mean and standard deviation of the counts per window, and of the totals per recording,
    over the given rows, added up 2,000 recordings at a time in double precision."""
    s1, s2 = np.zeros(C), np.zeros(C)                     # sums of counts and of squared counts per channel
    t1, t2 = np.zeros(C), np.zeros(C)                     # the same for the totals per recording
    n = 0                                                 # number of (recording, window) rows added
    for a in range(0, len(rows), 2000):
        b = X[rows[a:a + 2000]].astype(np.float64)        # 2,000 recordings as double-precision numbers
        s1 += b.sum(axis=(0, 1))
        s2 += (b ** 2).sum(axis=(0, 1))
        n += b.shape[0] * b.shape[1]
        totals = b.sum(axis=1)                            # every recording's total per channel
        t1 += totals.sum(0)
        t2 += (totals ** 2).sum(0)
    mu, mu_total = s1 / n, t1 / len(rows)                 # the means
    stats = {}
    stats['mu_window'] = mu.astype(np.float32)
    stats['sd_window'] = (np.sqrt(np.maximum(s2 / n - mu ** 2, 0)) + 1e-6).astype(np.float32)   # spread (+ a tiny
    stats['mu_total'] = mu_total.astype(np.float32)                                           # number, never 0)
    stats['sd_total'] = (np.sqrt(np.maximum(t2 / len(rows) - mu_total ** 2, 0)) + 1e-6).astype(np.float32)
    return stats


def input_statistics(X):
    """Average size of one test recording: windows, channels, spikes, non-empty cells, active channels."""
    spikes, cells, active = 0, 0, 0
    for a in range(0, len(X), 1000):                      # 1,000 recordings at a time
        part = X[a:a + 1000]
        spikes += int(part.sum(dtype=np.int64))           # input spikes
        cells += int(np.count_nonzero(part))              # (window, channel) cells with at least one spike
        active += int(np.count_nonzero(part.sum(axis=1, dtype=np.int64)))   # channels with at least one spike
    n = len(X)
    return {'windows': int(X.shape[1]), 'channels': int(X.shape[2]), 'spikes': spikes / n,
            'nonzero_cells': cells / n, 'nonzero_channels': active / n}


def window_orders(n, seed):
    """A random order of the 100 windows for each of n recordings (an n x 100 array of window numbers):
    sorting random numbers gives every recording its own random order."""
    rng = np.random.default_rng(seed)                     # a random generator with a fixed seed
    return np.argsort(rng.random((n, T)), axis=1)


def to_gpu(X, rows):
    """The recordings `rows` of table X as one tensor of 8-bit counts on the GPU."""
    out = torch.empty((len(rows), X.shape[1], X.shape[2]), dtype=torch.uint8, device=DEVICE)   # room on the GPU
    for a in range(0, len(rows), 4096):                   # copied 4,096 recordings at a time (saves memory)
        out[a:a + 4096] = torch.from_numpy(X[rows[a:a + 4096]]).to(DEVICE)
    return out


def orders_to_gpu(orders):
    """Window orders as a tensor on the GPU (or None when there are none)."""
    if orders is None:
        return None
    return torch.from_numpy(orders).to(DEVICE)


def get_batch(X, rows, order=None):
    """The recordings `rows` of a table on the GPU as float numbers; with `order`, each one's windows in
    its own order."""
    rows = torch.as_tensor(rows, device=X.device)         # the row numbers, on the GPU
    xb = X[rows]                                          # these recordings: (batch, windows, channels)
    if order is not None:                                 # window t of the result is window o[t] of the recording
        o = order[rows]                                   # the window order of each recording in the batch
        xb = torch.gather(xb, 1, o[:, :, None].expand(-1, -1, xb.shape[2]))   # take the windows in that order
    return xb.float()                                     # counts as float numbers, as the models expect


def thin(rows, n):
    """At most n of the rows, spread evenly (used only in a quick test)."""
    if len(rows) <= n:
        return rows
    return rows[np.linspace(0, len(rows) - 1, n).astype(np.int64)]

## Step 8. How energy is computed (computed, not measured)

No chip is measured. Energy per test recording follows the hardware-aware accounting of Dampfhoffer
et al. (2022): every weight and every neuron's state sits in on-chip memory, and every memory access
and every arithmetic operation costs energy. Memory accesses cost the most. Every model is charged on
the same imagined digital chip, with costs for 8-bit numbers in 45 nm technology based on Horowitz (2014):

| symbol | cost | what |
|---|---|---|
| m | 1.25 pJ | read or write one 8-bit number (a 64-bit read from an 8 KB memory, 10 pJ, divided by 8) |
| c | 0.23 pJ | one 8-bit multiply-add (multiplication 0.2 pJ + addition 0.03 pJ) |
| a | 0.03 pJ | one 8-bit addition |

From these, the cost of the operations the models do:

| operation | cost | pJ | what is charged |
|---|---|---|---|
| spike delivered to one synapse | 3m + a | 3.78 | read the weight; read and write the target neuron's state; one addition |
| first-layer multiply-add on a non-empty cell | 3m + c | 3.98 | read the weight; read and write the partial sum |
| any other multiply-add, no reuse | 4m + c | 5.23 | read the input and the weight; read and write the partial sum |
| multiply-add with ideal reuse | m + 4c | 2.17 | the weight is read once; the rest stays in registers |
| multiply-add, weights kept on chip | 4c + m/T | | the weight is read once per recording of T windows |
| spike with its weight in a register | 2m + c + a | 2.76 | |
| neuron state update | 2m + c | 2.73 | read the state, write it back, one multiply-add |
| spike added to a channel total | 2m + a | 2.53 | the count classifier |

Every model gets three numbers: **worst** (no data reuse), **within** (best case within a window: every
weight read once per window) and **across** (best case across the windows: the weights stay on the
chip for the whole recording). Models are compared within one case; a conclusion that holds in all
three cases does not depend on the hardware.

Per test recording: T windows, C channels, S input spikes, N non-empty (window, channel) cells.

**Conventional networks and order-blind designs.** W1 is the width of the first layer, P the
multiply-adds per window after the first layer, and Q the multiply-adds of a part that runs once per
recording. The input scaling folds into the first layer, so an empty cell costs only its read:

```
worst  = T C m + N W1 (3m + c) + (T P + Q)(4m + c)
within = T C m + (N W1 + T P + Q)(m + 4c)
across = T C m + (N W1 + T P)(4c + m / T) + Q (m + 4c)
```

**Spiking networks.** The first layer is charged either per input spike, S H (3m + a), or per
non-empty cell, N H (3m + c): the worst case takes the dearer of the two, the within case the cheaper
of S H (3m + a) and N H (m + 4c), the across case the cheaper of S H (2m + c + a) and N H (4c + m / T).
Every hidden spike costs 3m + a per synapse it reaches (2m + c + a across the windows), and every
neuron's state costs 2m + c per window.

In [ ]:
E_MEM = 1.25                                  # m: read or write one 8-bit number (pJ)
E_MAC = 0.23                                  # c: one multiply-add (pJ)
E_ADD = 0.03                                  # a: one addition (pJ)
COST_SPIKE = 3 * E_MEM + E_ADD                # a spike delivered to one synapse
COST_FIRST = 3 * E_MEM + E_MAC                # a first-layer multiply-add on a non-empty cell
COST_OTHER = 4 * E_MEM + E_MAC                # any other multiply-add without reuse
COST_REUSE = E_MEM + 4 * E_MAC                # a multiply-add with ideal reuse
COST_SPIKE_REG = 2 * E_MEM + E_MAC + E_ADD    # a spike whose weight is in a register
COST_UPDATE = 2 * E_MEM + E_MAC               # one neuron state update
COST_COUNT = 2 * E_MEM + E_ADD                # a spike added to a channel total
COSTS = {'memory_access': E_MEM, 'multiply_add': E_MAC, 'addition': E_ADD}   # saved with the experiment


def dense_energy(st, W1, P, Q):
    """Energy (pJ per recording) of a conventional network or order-blind design, in the three cases."""
    T, Cn, N = st['windows'], st['channels'], st['nonzero_cells']
    reads = T * Cn * E_MEM                                       # every input cell is read once
    kept = 4 * E_MAC + E_MEM / T                                 # a multiply-add with the weights kept on chip
    worst = reads + N * W1 * COST_FIRST + (T * P + Q) * COST_OTHER
    within = reads + (N * W1 + T * P + Q) * COST_REUSE
    across = reads + (N * W1 + T * P) * kept + Q * COST_REUSE
    parts = {'input_reads': reads, 'first_layer': N * W1 * COST_FIRST, 'other_layers': (T * P + Q) * COST_OTHER}
    return {'worst': worst, 'within': within, 'across': across, 'worst_is': 'no data reuse', 'parts': parts}


def spiking_energy(st, first_width, hidden_events, state_updates):
    """Energy (pJ per recording) of a spiking network.
    first_width: neurons every input spike reaches; hidden_events: hidden spikes x synapses each one
    reaches; state_updates: neuron states updated per recording."""
    T, S, N = st['windows'], st['spikes'], st['nonzero_cells']
    kept = 4 * E_MAC + E_MEM / T                                 # a multiply-add with the weights kept on chip
    per_spike = S * first_width * COST_SPIKE                      # first layer charged per input spike ...
    per_cell = N * first_width * COST_FIRST                       # ... or per non-empty cell
    hidden = hidden_events * COST_SPIKE                           # hidden spikes delivered
    state = state_updates * COST_UPDATE                           # neuron states updated
    worst = max(per_spike, per_cell) + hidden + state
    within = min(per_spike, N * first_width * COST_REUSE) + hidden + state
    across = min(S * first_width * COST_SPIKE_REG, N * first_width * kept) + hidden_events * COST_SPIKE_REG + state
    if per_spike >= per_cell:
        worst_is = 'per spike'
    else:
        worst_is = 'per cell'
    parts = {'first_layer_per_spike': per_spike, 'first_layer_per_cell': per_cell, 'hidden_spikes': hidden,
             'state_updates': state}
    return {'worst': worst, 'within': within, 'across': across, 'worst_is': worst_is, 'parts': parts}

## Step 9. The models

Every model takes a batch of tables, of shape (batch, 100 windows, 700 channels), and returns one score
per class; the class with the highest score is the answer. A model is built right after its seed is
set, and the order in which its layers are created fixes which random numbers start them, so a model
starts from the same weights whenever it is built with the same seed.

| model | kind | what it is |
|---|---|---|
| count_256 | counts only | channel totals -> 256 -> 35 classes |
| perminv | order-blind | every window 700 -> 128 -> 35, the scores of all windows added up |
| pool_128, pool_512 | order-blind | every window 700 -> 128 (or 512), averaged over windows, -> 128 -> 35 |
| meanmax_deep_128 (and _d20, _d50, _d70) | order-blind | a deeper per-window encoder, average and maximum |
| lstm_paper | conventional | the SSC authors' LSTM set-up: 128 cells, the answer read at the last window |
| lif_std, rlif | spiking | 128 leaky integrate-and-fire neurons, without and with recurrent weights |
| rnn | conventional | a plain recurrent network with exactly rlif's number of weights |

A name ending in `_d20`, `_d50` or `_d70` means dropout 0.2, 0.5 or 0.7 on the input while training
(dropout switches off a random share of the inputs, which helps a model generalise). Every model also
has an `energy` function that applies the formulas of step 8 to that model.

Most models start by scaling every channel: (count - mean) / spread, with the fixed numbers of
`scaling_stats`. They are stored with the model as "buffers": numbers that move to the GPU with the
model but are never trained.

### count_256: the floor

**Why.** It sees only each channel's total over the whole recording, so it loses both which channels
fire together within a moment and the order of the moments. Every other model should beat it.

**Architecture.** Channel totals, scaled -> 256 (ReLU) -> 35 classes.

**Energy.** Every input spike is added to its channel's total (2m + a); the 700 totals are read once;
the two layers run once per recording, on the non-empty totals.

In [ ]:
class CountClassifier(nn.Module):
    family = 'counts only'                                    # the kind of model (used by the analysis)

    def __init__(self, K, stats, h=256):
        super().__init__()                                    # the standard start of every PyTorch model
        self.net = nn.Sequential(nn.Linear(C, h), nn.ReLU(), nn.Linear(h, K))   # totals -> 256 -> K
        self.register_buffer('mu', torch.tensor(stats['mu_total']))           # mean total per channel
        self.register_buffer('sd', torch.tensor(stats['sd_total']))           # spread of the totals
        self.h, self.K = h, K                                 # sizes, remembered for the energy estimate

    def forward(self, x):                                     # x: (batch, windows, channels)
        totals = x.sum(1)                                     # add up the windows: one total per channel
        return self.net((totals - self.mu) / self.sd)         # scale the totals, then classify

    def energy(self, st, spikes=None, layer_spikes=None):
        reads = st['spikes'] * COST_COUNT + st['channels'] * E_MEM        # count every spike; read the totals
        macs1 = st['nonzero_channels'] * self.h                           # first layer, on non-empty totals
        macs2 = self.h * self.K                                           # second layer
        worst = reads + macs1 * COST_FIRST + macs2 * COST_OTHER           # no reuse
        best = reads + (macs1 + macs2) * COST_REUSE                       # runs once per recording: within and
        parts = {'input_reads': reads, 'first_layer': macs1 * COST_FIRST, 'other_layers': macs2 * COST_OTHER}
        return {'worst': worst, 'within': best, 'across': best, 'worst_is': 'no data reuse', 'parts': parts}   # across agree

### perminv: summed scores per window

**Why.** The simplest order-blind design, and the one with exactly lif_std's number of weights: it has
lif_std's two layers, but its units keep nothing from one window to the next. It is lif_std's
same-size partner.

**Architecture.** Every window: 700 -> 128 (ReLU) -> 35 scores. The scores of all windows are added up;
adding up does not depend on the order.

**Energy.** W1 = 128; P = 128 x 35 (the second layer runs in every window); Q = 0.

In [ ]:
class PerWindowScores(nn.Module):
    family = 'order-blind'

    def __init__(self, K, stats, h=H):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # mean count per channel and window
        self.register_buffer('sd', torch.tensor(stats['sd_window']))     # its spread
        self.net = nn.Sequential(nn.Linear(C, h), nn.ReLU(), nn.Linear(h, K))   # used on every window
        self.h, self.K = h, K                                 # sizes, remembered for the energy estimate

    def forward(self, x):
        x = (x - self.mu) / self.sd                           # scale every channel
        return self.net(x).sum(1)                             # scores of every window, added up

    def energy(self, st, spikes=None, layer_spikes=None):
        return dense_energy(st, W1=self.h, P=self.h * self.K, Q=0)

### pool_128 and pool_512: mean pooling

**Why.** The features of every window are averaged. pool_128 has exactly the weights of rlif and of
rnn; pool_512 has the SSC authors' LSTM's number of weights within 0.1%. They are the same-size partners
of those networks.

**Architecture.** Every window 700 -> h1 (ReLU); the average over windows; h1 -> 128 (ReLU) -> 35.

**Energy.** W1 = h1; P = 0; Q = h1 x 128 + 128 x 35.

In [ ]:
class MeanPool(nn.Module):
    family = 'order-blind'

    def __init__(self, K, stats, h1, h2):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # fixed scaling numbers
        self.register_buffer('sd', torch.tensor(stats['sd_window']))
        self.enc = nn.Sequential(nn.Linear(C, h1), nn.ReLU())            # every window -> h1 features
        self.dec = nn.Sequential(nn.Linear(h1, h2), nn.ReLU(), nn.Linear(h2, K))   # the average -> classes
        self.h1, self.h2, self.K = h1, h2, K                  # sizes, remembered for the energy estimate

    def forward(self, x):
        features = self.enc((x - self.mu) / self.sd)          # (batch, windows, h1)
        return self.dec(features.mean(1))                     # average over the windows, then classify

    def energy(self, st, spikes=None, layer_spikes=None):
        return dense_energy(st, W1=self.h1, P=0, Q=self.h1 * self.h2 + self.h2 * self.K)

### meanmax_deep_128 (with input dropout: _d20, _d50, _d70)

**Why.** A deeper per-window encoder, pooled by average and maximum: one of the strongest order-blind
designs on SHD, so it is the main order-blind candidate here.

**Architecture.** Every window 700 -> 128 -> 128 (ReLU after each); average and maximum over windows
(256) -> 128 (ReLU) -> 35. Optional dropout p on the scaled input while training.

**Energy.** W1 = 128; P = 128 x 128; Q = 256 x 128 + 128 x 35.

In [ ]:
class MeanMaxDeep(nn.Module):
    family = 'order-blind'

    def __init__(self, K, stats, h=H, p_in=0.0):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # fixed scaling numbers
        self.register_buffer('sd', torch.tensor(stats['sd_window']))
        self.enc = nn.Sequential(nn.Linear(C, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU())   # two layers per window
        self.dec = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, K))          # average and max -> classes
        if p_in > 0:
            self.drop = nn.Dropout(p_in)                      # dropout on the input while training
        else:
            self.drop = None
        self.h, self.K = h, K                                 # sizes, remembered for the energy estimate

    def forward(self, x):
        x = (x - self.mu) / self.sd                           # scale every channel
        if self.drop is not None:
            x = self.drop(x)
        f = self.enc(x)                                       # features of every window
        return self.dec(torch.cat([f.mean(1), f.max(1).values], 1))

    def energy(self, st, spikes=None, layer_spikes=None):
        return dense_energy(st, W1=self.h, P=self.h * self.h, Q=2 * self.h * self.h + self.h * self.K)

### lstm_paper: the SSC authors' LSTM set-up (Cramer et al. 2020)

**Why.** The published LSTM on SSC, set up as described there: one layer of 128 cells, dropout 0.2 on
the input and on the recurrent state, Adamax, 10 ms windows, and the answer read at the last window.

**Architecture.** Written out step by step with the defaults of Keras 2.3.0: input weights Glorot
uniform, recurrent weights orthogonal, one bias per gate unit (zero, except the forget gate's at 1); the
gates in the order input, forget, candidate, output. While training, one input-dropout mask per
recording is kept for every window, and one recurrent-dropout mask per recording is applied to the
previous output. A linear readout (Glorot uniform, zero bias) reads the output at the last window.

An LSTM cell, in every window: the four gates are computed from the input and the last output; the
cell's memory c = forget gate x c + input gate x candidate; the output h = output gate x tanh(c).

**Energy.** W1 = 4 x 128; P = 4 x 128 x 128 (the gates' recurrent weights); Q = 128 x 35 (the readout runs once).

In [ ]:
class LSTMPaper(nn.Module):
    family = 'conventional'

    def __init__(self, K, stats, p_in=0.2, p_rec=0.2, h=H):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # fixed scaling numbers
        self.register_buffer('sd', torch.tensor(stats['sd_window']))
        self.p_in, self.p_rec, self.h, self.K = p_in, p_rec, h, K
        self.W = nn.Linear(C, 4 * h)                          # input weights of the four gates
        self.U = nn.Linear(h, 4 * h, bias=False)              # recurrent weights of the four gates
        self.fc = nn.Linear(h, K)                             # readout
        nn.init.xavier_uniform_(self.W.weight)                # Glorot uniform (Keras's default)
        nn.init.orthogonal_(self.U.weight)                    # orthogonal (Keras's default)
        nn.init.zeros_(self.W.bias)                           # zero biases ...
        with torch.no_grad():
            self.W.bias[h:2 * h] = 1.0                        # ... except the forget gate's, at 1
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        x = (x - self.mu) / self.sd                           # scale every channel
        n = x.shape[0]                                        # recordings in the batch
        if self.training and self.p_in > 0:                   # input dropout: one mask per recording, the same
            keep = torch.bernoulli(x.new_full((n, 1, x.shape[2]), 1 - self.p_in))   # for every window
            x = x * keep / (1 - self.p_in)
        zx = self.W(x)                                        # the input part of the gates, for every window
        hs = x.new_zeros(n, self.h)                           # the output, starting at 0
        c = x.new_zeros(n, self.h)                            # the cell's memory, starting at 0
        mask = None
        if self.training and self.p_rec > 0:                  # recurrent dropout: one mask per recording
            mask = torch.bernoulli(x.new_full((n, self.h), 1 - self.p_rec)) / (1 - self.p_rec)
        for t in range(x.shape[1]):                           # the windows in order
            if mask is None:
                z = zx[:, t] + self.U(hs)
            else:
                z = zx[:, t] + self.U(hs * mask)
            i, f, g, o = z.chunk(4, 1)                        # input, forget, candidate and output gates
            c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)   # update the memory
            hs = torch.sigmoid(o) * torch.tanh(c)             # the output of this window
        return self.fc(hs)                                    # the answer, read at the last window

    def energy(self, st, spikes=None, layer_spikes=None):
        return dense_energy(st, W1=4 * self.h, P=4 * self.h * self.h, Q=self.h * self.K)

### lif_std and rlif: small spiking networks

**Why.** The small spiking networks of the study. lif_std's only memory is each neuron's leaky voltage;
rlif adds recurrent weights among its neurons. Together with rnn (the same weights in a conventional
recurrent network) they make spiking and conventional twins of equal size.

**Architecture.** 128 leaky integrate-and-fire neurons. In every window, in order:

```
voltage = beta x voltage + input weights x counts [+ recurrent weights x last spikes] - last spike x 1
spike   = 1 if voltage > 1, else 0
```

A neuron that spiked loses 1 from its voltage in the next window. A readout (128 -> 35) is applied to
every window's spikes, and the results are added up. This is snnTorch 1.0.0's Leaky and RLeaky neuron
written out step by step (the self-checks confirm that the two agree exactly). A spike is a jump from 0
to 1, whose slope is zero almost everywhere, so training uses a smooth stand-in for the slope: the slope
of a "fast sigmoid", 1 / (25 |x| + 1)^2, as snnTorch does.

**Energy.** The first layer per input spike or per non-empty cell (128 neurons reached); every hidden
spike reaches the 35 readout units (and, in rlif, the 128 neurons of the layer); 128 state updates per window.

In [ ]:
class SpikeFastSigmoid(torch.autograd.Function):
    """A spike (1) where the input is above 0, else 0; when training, the slope of a fast sigmoid."""

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)                              # remember the input for the backward pass
        return (x > 0).float()                                # 1 where above the threshold, else 0

    @staticmethod
    def backward(ctx, grad):
        (x,) = ctx.saved_tensors
        return grad / (25 * torch.abs(x) + 1.0) ** 2          # the smooth stand-in slope


class LIFNet(nn.Module):
    family = 'spiking'

    def __init__(self, K, beta, stats, recurrent=False, h=H):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # fixed scaling numbers
        self.register_buffer('sd', torch.tensor(stats['sd_window']))
        self.fc1 = nn.Linear(C, h)                            # input weights: 700 channels -> 128 neurons
        self.fc2 = nn.Linear(h, K)                            # readout: 128 neurons -> 35 classes
        if recurrent:
            self.recurrent = nn.Linear(h, h)                  # rlif only: weights from the neurons to themselves
        else:
            self.recurrent = None
        self.register_buffer('beta', torch.as_tensor(beta, dtype=torch.float32))   # share of voltage kept
        self.register_buffer('threshold', torch.as_tensor(1.0))                     # spike when above 1
        self.h, self.K = h, K                                 # sizes, remembered for the energy estimate
        self.counting = False                                 # True while testing: count the hidden spikes
        self.spike_total = 0.0                                # hidden spikes counted so far

    def forward(self, x):
        x = (x - self.mu) / self.sd                           # scale every channel
        beta = self.beta.clamp(0, 1)                          # beta must stay between 0 and 1
        mem = torch.zeros(x.shape[0], self.h, device=x.device)   # every neuron's voltage, starting at 0
        spk = torch.zeros_like(mem)                           # every neuron's spike in the last window
        out = 0                                               # the class scores, added up over windows
        n_spikes = 0                                          # hidden spikes of this batch
        for t in range(x.shape[1]):                           # the windows IN ORDER: here order can matter
            reset = spk.detach()                              # neurons that spiked in the last window lose 1
            if self.recurrent is None:
                mem = beta * mem + self.fc1(x[:, t]) - reset * self.threshold
            else:
                mem = beta * mem + self.fc1(x[:, t]) + self.recurrent(spk) - reset * self.threshold
            spk = SpikeFastSigmoid.apply(mem - self.threshold)   # spike where the voltage is above 1
            out = out + self.fc2(spk)                         # readout of this window, added up
            if self.counting:
                n_spikes = n_spikes + spk.detach().sum()      # count this window's spikes
        if self.counting:
            self.spike_total += float(n_spikes)
        return out

    def energy(self, st, spikes, layer_spikes=None):
        fan_out = self.K                                      # every hidden spike reaches the 35 readout units
        if self.recurrent is not None:
            fan_out = fan_out + self.h                        # ... and, in rlif, the 128 neurons of the layer
        return spiking_energy(st, first_width=self.h, hidden_events=spikes * fan_out,
                              state_updates=self.h * st['windows'])   # one state per neuron per window

### rnn: a plain recurrent network

**Why.** A conventional network that reads the windows in order, with exactly rlif's number of weights:
rlif's conventional twin.

**Architecture.** The scaled input; PyTorch's RNN (tanh units), 700 -> 128; a readout 128 -> 35 at every
window, added up over the windows.

**Energy.** W1 = 128; P = 128 x 128 + 128 x 35; Q = 0.

In [ ]:
class RecurrentNet(nn.Module):
    family = 'conventional'

    def __init__(self, kind, K, stats, p_in=0.0, h=H):
        super().__init__()
        self.register_buffer('mu', torch.tensor(stats['mu_window']))     # fixed scaling numbers
        self.register_buffer('sd', torch.tensor(stats['sd_window']))
        self.drop = nn.Dropout(p_in)                          # dropout on the input (p_in = 0: none)
        if kind == 'rnn':
            self.rec = nn.RNN(C, h, batch_first=True)         # a plain recurrent layer (tanh)
        else:
            self.rec = nn.LSTM(C, h, batch_first=True)        # an LSTM layer
        self.fc = nn.Linear(h, K)                             # readout
        self.kind, self.h, self.K = kind, h, K                # remembered for the energy estimate

    def forward(self, x):
        x = self.drop((x - self.mu) / self.sd)                # scale, then dropout (only while training)
        hidden, _ = self.rec(x)                               # the hidden state after every window, in order
        return self.fc(hidden).sum(1)                         # readout of every window, added up

    def energy(self, st, spikes=None, layer_spikes=None):
        if self.kind == 'rnn':
            gates = 1
        else:
            gates = 4                                         # an LSTM has four gates
        return dense_energy(st, W1=gates * self.h, P=gates * self.h * self.h + self.h * self.K, Q=0)

### Building a model by its name

`make_model` builds a fresh, untrained model from its name.

In [ ]:
def make_model(name, stats):
    if name == 'count_256':
        return CountClassifier(K, stats, 256)
    if name == 'perminv':
        return PerWindowScores(K, stats, H)
    if name == 'pool_128':
        return MeanPool(K, stats, H, H)
    if name == 'pool_512':
        return MeanPool(K, stats, 4 * H, H)
    if name == 'meanmax_deep_128':
        return MeanMaxDeep(K, stats, H, 0.0)
    if name in ('meanmax_deep_128_d20', 'meanmax_deep_128_d50', 'meanmax_deep_128_d70'):
        return MeanMaxDeep(K, stats, H, int(name[-2:]) / 100)     # the last two digits: d20 -> dropout 0.2
    if name == 'lstm_paper':
        return LSTMPaper(K, stats)
    if name == 'lif_std':
        return LIFNet(K, BETA, stats, recurrent=False)
    if name == 'rlif':
        return LIFNet(K, BETA, stats, recurrent=True)
    if name == 'rnn':
        return RecurrentNet('rnn', K, stats, 0.0)
    raise ValueError('unknown model: ' + name)


def n_weights(model):
    """The number of trainable numbers (weights and biases) of a model."""
    return sum(p.numel() for p in model.parameters())

## Step 10. Training and testing

**Training.** A fresh model sees the fitting recordings in batches of 128, in a new random order of the
recordings every epoch (never of the windows). After each batch the optimiser changes the weights a
little to lower the cross-entropy loss (a measure of how wrong the answers are). The last epoch is kept.

**Testing.** The model answers every recording; the accuracy is the share of right answers. For
spiking networks the hidden spikes are counted, because the energy estimate needs them.

In [ ]:
def test_model(model, X, y, order, seed):
    """Accuracy, the answers, and (spiking networks) the hidden spikes per recording."""
    model.eval()                                              # testing mode: no dropout
    if model.family == 'spiking':                             # start counting the hidden spikes
        model.counting = True
        model.spike_total = 0.0
    answers = []
    with torch.no_grad():                                     # testing only: slopes are not needed
        with torch.random.fork_rng(devices=[]):               # remember the CPU's random state ...
            torch.default_generator.manual_seed(seed)         # ... use a fixed seed for this test ...
            for i in range(0, len(y), BATCH):                 # ... (the old state comes back afterwards)
                rows = np.arange(i, min(i + BATCH, len(y)))   # the next 128 recordings
                scores = model(get_batch(X, rows, order))     # one score per class
                answers.append(scores.argmax(1).cpu().numpy())    # the class with the highest score
    answers = np.concatenate(answers)                         # all answers in one array
    spikes = None
    if model.family == 'spiking':
        model.counting = False
        spikes = model.spike_total / len(y)                   # hidden spikes per recording
    accuracy = float((answers == y).mean())                   # share of right answers
    return accuracy, answers, spikes


def train_model(model, X, y, order, label=''):
    """Train a model by the recipe: Adamax, the last epoch kept."""
    optimizer = torch.optim.Adamax(model.parameters(), lr=RECIPE['lr'])
    loss_fn = nn.CrossEntropyLoss()                           # how wrong the answers are
    for epoch in tqdm(range(RECIPE['epochs']), desc=label, leave=False, unit='epoch'):   # progress bar over epochs
        model.train()                                         # training mode (dropout on)
        shuffled = np.random.permutation(len(y))              # a new random order of the recordings
        for i in range(0, len(shuffled), BATCH):
            rows = shuffled[i:i + BATCH]                      # the recordings of this batch
            xb = get_batch(X, rows, order)                    # their tables
            yb = torch.from_numpy(y[rows]).to(DEVICE)         # their classes
            optimizer.zero_grad()                             # forget the slopes of the last batch
            loss = loss_fn(model(xb), yb)                     # how wrong the model is on this batch
            loss.backward()                                   # the slope of the loss for every weight
            optimizer.step()                                  # change the weights a little

## Step 11. Self-checks

Quick tests that stop the notebook if something is wrong:

- every order-blind model gives exactly the same answer when the windows are shuffled;
- perminv has exactly lif_std's number of weights, and pool_128 exactly rlif's and rnn's;
- lif_std and rlif give the same weights, scores, spike counts and slopes as snnTorch's Leaky and RLeaky;
- lstm_paper gives the same scores as PyTorch's own LSTM with the same weights, and its forget-gate
  biases start at 1.

In [ ]:
def check_order_blind(names, stats, x, x_shuffled):
    """Order-blind models must ignore the order of the windows; same-size partners must match in size."""
    sizes = {}
    for name in names:
        torch.manual_seed(0)
        model = make_model(name, stats).to(DEVICE)
        with torch.no_grad():
            model.eval()
            out = model(x)                                    # scores with the windows in order
            same = bool(torch.allclose(out, model(x_shuffled), atol=1e-4))   # ... and shuffled
        sizes[name] = n_weights(model)
        if model.family in ('order-blind', 'counts only'):
            assert same, name + ' should ignore the order of the windows'
    for a, b in [('perminv', 'lif_std'), ('pool_128', 'rlif'), ('pool_128', 'rnn')]:
        if a in sizes and b in sizes:
            assert sizes[a] == sizes[b], a + ' and ' + b + ' should have the same number of weights'
    print('   the order-blind models ignore the window order; the same-size partners match')


def check_against_snntorch(stats, x, y):
    """lif_std and rlif against snnTorch's Leaky and RLeaky, built after the same seed (on the CPU)."""
    import snntorch as snn
    from snntorch import surrogate
    xc, yc = x.cpu(), torch.as_tensor(y).cpu()
    for recurrent in (False, True):
        torch.manual_seed(0)
        fc1, fc2 = nn.Linear(C, H), nn.Linear(H, K)           # snnTorch's version, layer by layer
        if recurrent:
            lif = snn.RLeaky(beta=BETA, linear_features=H, spike_grad=surrogate.fast_sigmoid())
        else:
            lif = snn.Leaky(beta=BETA, spike_grad=surrogate.fast_sigmoid())
        torch.manual_seed(0)
        own = LIFNet(K, BETA, stats, recurrent=recurrent)     # this notebook's version
        xs = (xc - torch.tensor(stats['mu_window'])) / torch.tensor(stats['sd_window'])
        out_ref, spikes_ref = 0, 0.0
        if recurrent:
            spk, mem = lif.init_rleaky()
        else:
            mem = lif.init_leaky()
        for t in range(xs.shape[1]):                          # snnTorch, window by window
            if recurrent:
                spk, mem = lif(fc1(xs[:, t]), spk, mem)
            else:
                spk, mem = lif(fc1(xs[:, t]), mem)
            out_ref = out_ref + fc2(spk)
            spikes_ref += spk.sum().item()
        own.counting = True
        out_own = own(xc)                                     # this notebook's version
        w_ref = [fc1.weight, fc1.bias, fc2.weight, fc2.bias]
        w_own = [own.fc1.weight, own.fc1.bias, own.fc2.weight, own.fc2.bias]
        if recurrent:
            w_ref += [lif.recurrent.weight, lif.recurrent.bias]
            w_own += [own.recurrent.weight, own.recurrent.bias]
        same_w = all(torch.equal(a, b) for a, b in zip(w_ref, w_own))            # same weights?
        nn.CrossEntropyLoss()(out_ref, yc).backward()
        nn.CrossEntropyLoss()(out_own, yc).backward()
        same_g = all(torch.equal(a.grad, b.grad) for a, b in zip(w_ref, w_own))  # same slopes?
        ok = same_w and torch.equal(out_ref, out_own) and spikes_ref == own.spike_total and same_g
        if recurrent:
            name = 'rlif'
        else:
            name = 'lif_std'
        print(f'   {name} against snnTorch {snn.__version__}: identical weights, scores, spike counts and slopes: {ok}')
        assert ok, 'the spiking neurons here differ from snnTorch 1.0.0'


def check_lstm_paper(stats, x):
    """lstm_paper without dropout against PyTorch's LSTM with the same weights (on the CPU)."""
    model = LSTMPaper(K, stats, 0.0, 0.0).eval()              # no dropout
    ref = nn.LSTM(C, H, batch_first=True)                     # PyTorch's own LSTM
    xc = x.cpu()
    with torch.no_grad():
        ref.weight_ih_l0.copy_(model.W.weight)                # give it the same weights
        ref.weight_hh_l0.copy_(model.U.weight)
        ref.bias_ih_l0.copy_(model.W.bias)
        ref.bias_hh_l0.zero_()
        scaled = (xc - model.mu) / model.sd
        same = bool(torch.allclose(model.fc(ref(scaled)[0][:, -1]), model(xc), atol=1e-4))
    forget = bool((model.W.bias[H:2 * H] == 1).all())         # forget-gate biases start at 1?
    assert n_weights(model) == 4 * H * (C + H + 1) + H * K + K
    print(f'   lstm_paper against PyTorch\'s LSTM: same scores {same}; forget-gate biases start at 1: {forget}')
    assert same and forget

## Step 12. Saving results and estimating the time

Every run is saved as one small file the moment it ends. It is written under a temporary name and then
renamed, so a file is either complete or absent. The time estimate times a few training steps of each
model on this machine: minutes per run = seconds per step x steps per epoch x epochs x 1.15 (testing
adds about 15%).

In [ ]:
def save_json(path, obj):
    """Save a dictionary as a .json file (written in one step)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + '.tmp', 'w') as f:
        json.dump(obj, f)
    os.replace(path + '.tmp', path)


def gpu_name():
    if DEVICE == 'cuda':
        return torch.cuda.get_device_name(0)
    return 'CPU'


def minutes_per_run(name, stats, X, y):
    """Estimated minutes for one run of a model: a few training steps are timed on real batches."""
    torch.manual_seed(0)
    model = make_model(name, stats).to(DEVICE)
    model.train()
    optimizer = torch.optim.Adamax(model.parameters(), lr=RECIPE['lr'])
    loss_fn = nn.CrossEntropyLoss()
    for step in range(4):                                     # step 0 warms up; steps 1 to 3 are timed
        if step == 1:
            if DEVICE == 'cuda':
                torch.cuda.synchronize()                      # wait until the GPU has finished
            start = time.time()
        rows = np.random.default_rng(step % 2).choice(len(y), min(BATCH, len(y)), replace=False)
        optimizer.zero_grad()
        loss_fn(model(get_batch(X, rows)), torch.from_numpy(y[rows]).to(DEVICE)).backward()
        optimizer.step()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    seconds_per_step = (time.time() - start) / 3
    steps_per_epoch = int(np.ceil(len(y) / BATCH))
    return seconds_per_step * steps_per_epoch * RECIPE['epochs'] * 1.15 / 60


# the overall progress bar counts estimated minutes
BAR_FORMAT = '{l_bar}{bar}| {n:.1f}/{total:.1f} estimated min [{elapsed} elapsed, about {remaining} left]'

## Step 13. Count the spikes and prepare the data

Counting the spikes of the 75,466 training recordings takes several minutes. Then 5% of the training
recordings are held out, the scaling numbers and the input statistics are computed, and the tables go
to the GPU as 8-bit counts (about 6.7 GB). The tables in the computer's memory are deleted afterwards.
A quick test counts only every 20th training recording and every 10th test recording.

In [ ]:
if QUICK:
    EVERY_TRAIN, EVERY_TEST = 20, 10                          # a quick test: a small share of the recordings
else:
    EVERY_TRAIN, EVERY_TEST = 1, 1                            # the full study: every recording
X_train, y_train = count_spikes(SSC_TRAIN, EVERY_TRAIN)
X_test, y_test = count_spikes(SSC_TEST, EVERY_TEST)

fit_rows, val_rows = holdout(y_train, HOLDOUT, VAL_SEED)
test_rows = np.arange(len(y_test))
if QUICK:                                                     # a quick test: a few recordings only
    fit_rows, val_rows, test_rows = thin(fit_rows, 384), thin(val_rows, 128), thin(test_rows, 256)
stats = scaling_stats(X_train, fit_rows)                      # from the fitting recordings only
input_stats = input_statistics(X_test[test_rows])             # the test recordings' size, for the energy
X_fit, y_fit = to_gpu(X_train, fit_rows), y_train[fit_rows]
X_val, y_val = to_gpu(X_train, val_rows), y_train[val_rows]
X_tst, y_tst = to_gpu(X_test, test_rows), y_test[test_rows]
del X_train, X_test                                           # free the computer's memory
check_rows = np.linspace(0, len(fit_rows) - 1, min(TRAIN_CHECK, len(fit_rows))).astype(np.int64)
print(f'fitting {len(fit_rows)}, held out {len(val_rows)}, test {len(test_rows)} recordings of {T} windows x {C} channels')
if DEVICE == 'cuda':
    print(f'GPU memory still free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

## Step 14. Train every model

The designs are trained once per seed; every network is trained in order (`q = 0`) and on shuffled
windows (`q = 1`), three seeds each. In the shuffled arm, every recording's windows are in its own
random order, for the fitting, held-out and test recordings alike. After training, every run records
the accuracy on 5,000 fitting recordings, on the held-out and on the test recordings, the hidden spikes
and the energy estimate. The cell first saves a description of the experiment, runs the self-checks
and prints the time estimate.

In [ ]:
experiment = 'ssc_10ms'
folder = RESULTS_DIR + experiment + '/'
meta = {'experiment': experiment, 'label': 'SSC, 100 windows of 10 ms', 'dataset': 'SSC', 'window_mode': 'fixed',
        'windows': T, 'window_ms': 1000 * WIDTH, 'channels': C, 'classes': K, 'chance': 1 / K,
        'competence_bar': 1 / K + 0.5 * (1 - 1 / K),  # training accuracy halfway from chance to perfect
        'seeds': SEEDS_USED, 'holdout_share': HOLDOUT, 'n_fit': len(y_fit), 'n_val': len(y_val), 'n_test': len(y_tst),
        'beta': BETA, 'input_per_test_recording': input_stats, 'test_labels': y_tst.tolist(),
        'designs': DESIGNS, 'networks': NETWORKS, 'energy_costs_pJ': COSTS, 'quick': QUICK, 'models': {},
        'environment': {'torch': torch.__version__, 'device': gpu_name()}}   # what the runs were computed on
for name in DESIGNS + NETWORKS:                               # every model's kind and size
    torch.manual_seed(0)
    model = make_model(name, stats)
    if name in DESIGNS:
        kind = 'design'
    else:
        kind = 'network'
    meta['models'][name] = {'kind': kind, 'family': model.family, 'weights': n_weights(model),
                            'recipe': RECIPE, 'scale': '128 units'}
save_json(folder + 'meta.json', meta)

if RUN_CHECKS:
    print('self-checks')
    x = get_batch(X_tst, np.arange(32))                       # 32 test recordings in order ...
    x_shuffled = get_batch(X_tst, np.arange(32), orders_to_gpu(window_orders(len(y_tst), 5)))   # ... and shuffled
    check_order_blind(DESIGNS + NETWORKS, stats, x[:4], x_shuffled[:4])
    check_against_snntorch(stats, x, y_tst[:32])
    check_lstm_paper(stats, x[:4])

jobs = []                                                     # every run: (model, q, seed)
for name in DESIGNS:
    for seed in SEEDS_USED:
        jobs.append((name, 0, seed))
for q in (0, 1):
    for name in NETWORKS:
        for seed in SEEDS_USED:
            jobs.append((name, q, seed))
to_do = []                                                    # the runs not saved yet
for job in jobs:
    if not os.path.exists(folder + f'runs/{job[0]}__q{job[1]}__s{job[2]}.json'):
        to_do.append(job)
print(f'{experiment}: {len(jobs)} runs, {len(to_do)} still to do')

minutes = {}                                                  # the time estimate
total_hours = 0.0
for name in DESIGNS + NETWORKS:
    n = 0
    for job in to_do:
        if job[0] == name:
            n += 1
    if n > 0:
        minutes[name] = minutes_per_run(name, stats, X_fit, y_fit)
        total_hours += minutes[name] * n / 60
        print(f'   {name:24s} {minutes[name]:7.1f} min per run x {n} runs = {minutes[name] * n / 60:6.2f} h')
print(f'   estimated total: about {total_hours:.1f} hours on {gpu_name()}')

total = 0.0
for job in to_do:
    total += minutes[job[0]]
bar = tqdm(total=total, desc=experiment, bar_format=BAR_FORMAT)   # the progress bar of the experiment
for name, q, seed in to_do:
    start = time.time()
    order_fit, order_val, order_test = None, None, None
    if q == 1:                                                # the shuffled arm: one random order per recording
        order_fit = orders_to_gpu(window_orders(len(y_fit), SHUFFLE_SEED_FIT + 10 * seed))
        order_val = orders_to_gpu(window_orders(len(y_val), SHUFFLE_SEED_VAL + 10 * seed))
        order_test = orders_to_gpu(window_orders(len(y_tst), SHUFFLE_SEED_TEST + 10 * seed))
    torch.manual_seed(seed)                                   # the seed fixes the starting weights ...
    np.random.seed(seed)                                      # ... and the order of the recordings in training
    model = make_model(name, stats).to(DEVICE)
    train_model(model, X_fit, y_fit, order_fit, label=f'{name} q{q} s{seed}')
    eval_seed = EVAL_SEED + seed
    X_check = X_fit[torch.as_tensor(check_rows, device=X_fit.device)]   # the 5,000 fitting recordings checked
    order_check = None
    if order_fit is not None:
        order_check = order_fit[torch.as_tensor(check_rows, device=order_fit.device)]   # in their own orders
    train_acc = test_model(model, X_check, y_fit[check_rows], order_check, eval_seed)[0]
    val_acc = test_model(model, X_val, y_val, order_val, eval_seed)[0]
    test_acc, answers, spikes = test_model(model, X_tst, y_tst, order_test, eval_seed)
    record = {'experiment': experiment, 'model': name, 'q': q, 'seed': seed, 'recipe': RECIPE,
              'train_acc': train_acc, 'val_acc': val_acc, 'test_acc': test_acc, 'preds': answers.tolist(),
              'spikes_per_recording': spikes, 'layer_spikes_per_recording': None,
              'energy_pJ': model.energy(input_stats, spikes), 'best_epoch': None, 'val_curve': None,
              'weights': n_weights(model), 'device': gpu_name(), 'seconds': time.time() - start}
    save_json(folder + f'runs/{name}__q{q}__s{seed}.json', record)
    bar.update(min(minutes[name], bar.total - bar.n))
    e = record['energy_pJ']
    tqdm.write(f'{name:22s} q{q} s{seed}: train {train_acc:.3f}  val {val_acc:.4f}  test {test_acc:.4f}'
               f'  | energy {e["worst"] / 1e6:.3g} / {e["within"] / 1e6:.3g} / {e["across"] / 1e6:.3g} uJ  | {record["seconds"]:.0f} s')
    if train_acc < meta['competence_bar'] and not QUICK:
        tqdm.write(f'   note: training accuracy below the competence bar ({meta["competence_bar"]:.4f}): reported, not interpreted')
    del model, X_check
bar.n = bar.total                                             # the estimate is approximate: show the bar full
bar.close()

## Step 15. Where the results are

A full run's results are on your Google Drive. A quick test's results are on this Colab machine, which
is deleted when the session ends, so they are packed into one `.zip` file and offered as a download.
When `02_train_SHD.ipynb` has finished too, `04_analyze_results.ipynb` makes the tables and figures
from your own runs (with `RESULTS = 'mine'` in its settings).

In [ ]:
if USE_DRIVE:
    print('The results are on your Google Drive, in', RESULTS_DIR)
else:
    archive = shutil.make_archive(RESULTS_DIR.rstrip('/'), 'zip', RESULTS_DIR)   # one .zip file of the folder
    print('The results are in', archive)
    try:
        from google.colab import files        # Colab's tool for saving a file to your computer
        files.download(archive)               # your browser saves the .zip file
    except ImportError:                       # not on Colab: the file stays where it is
        pass